<a href="https://colab.research.google.com/github/CharalapML/ColabCode/blob/main/Aug2026_TEST_for_WORKS_Ringing_Up_VIDEO_v6_Transparent_Bell_Swinging_Dual_canvas_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
!pip install cairosvg

In [ ]:
import cv2
import numpy as np
from moviepy.editor import ImageSequenceClip
import os
import matplotlib.pyplot as plt
import skimage as ski
import io

import glob
import cv2
import cairosvg
from xml.etree import ElementTree as ET

import math

In [ ]:
from google.colab import files

if os.path.exists('video_v6.mp4'):
    files.download('video_v6.mp4')
else:
    print('File not found.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
ET.register_namespace("", "http://www.w3.org/2000/svg")

bg_root  = ET.parse('/content/gdrive/My Drive/Images_png_svg_jpg_etc/FrameOnly.svg').getroot()
obj_root = ET.parse('/content/gdrive/My Drive/Images_png_svg_jpg_etc/BellinWheel.svg').getroot()

# Extract viewBox dimensions for bell_pivot_local_x and bell_pivot_local_y
viewbox_str = obj_root.get("viewBox")
bell_width  = 100
bell_height = 100
if viewbox_str:
    try:
        _, _, bell_width, bell_height = map(float, viewbox_str.split())
        # Using top-center of the bell as the local pivot point
        bell_pivot_local_x = bell_width / 2
        bell_pivot_local_y = 0
    except ValueError:
        print("Warning: Bell.svg viewBox parsing error, using (0,0) as pivot.")
        bell_pivot_local_x = bell_width / 2   #  0
        bell_pivot_local_y = 0
else:
    print("Warning: Bell.svg viewBox not found, using (0,0) as pivot.")
    bell_pivot_local_x = bell_width / 2   #  0
    bell_pivot_local_y = 0

# bell_pivot_local_x = 40
# bell_pivot_local_y = 0

print('bell_width = ', bell_width, 'bell_height = ', bell_height)
print('bell_pivot_local_x = ', bell_pivot_local_x, 'bell_pivot_local_y = ', bell_pivot_local_y)
print('viewbox_str = ', viewbox_str)

scene_root = ET.Element(bg_root.tag, bg_root.attrib)

bg_group = ET.SubElement(scene_root, "{http://www.w3.org/2000/svg}g")
bg_group.set("id", "background")

obj_group = ET.SubElement(scene_root, "{http://www.w3.org/2000/svg}g")
obj_group.set("id", "movable_object")

bg_group.extend(list(bg_root))
obj_group.extend(list(obj_root))

# Define a scale factor for the background
bg_scale_factor = 0.7  # You can change this value (e.g., 0.25 for 25% size)

# Apply the scale transformation to the background group
bg_group.set("transform", f"scale({bg_scale_factor})")

ET.ElementTree(scene_root).write(
    "scene.svg",
    encoding="utf-8",
    xml_declaration=True
)

bell_width =  100 bell_height =  100
bell_pivot_local_x =  50.0 bell_pivot_local_y =  0
viewbox_str =  None


In [ ]:
width_attr  = obj_root.get("width")
height_attr = obj_root.get("height")

if width_attr and height_attr:
    bell_width  = float(width_attr.replace("px", ""))
    bell_height = float(height_attr.replace("px", ""))
else:
    bell_width = bell_height = 100  # fallback


bell_pivot_local_x = (bell_width / 2)
bell_pivot_local_y = (bell_height / 2) # - 30  # 120 #32

print(width_attr, height_attr)
bell_pivot_local_x, bell_pivot_local_y

1600 1613


(800.0, 806.5)

In [ ]:

tree = ET.parse("scene.svg")
root = tree.getroot()

SVG_NS = "http://www.w3.org/2000/svg"

for el in root.iter(f"{{{SVG_NS}}}g"):
    if el.get("id") == "movable_object":
        el.set("transform", "translate(200,100)")

tree.write("scene_moved.svg")


# **The following puts a delay at the end of each cycle**



In [ ]:
object_spin_angle_deg = [0,31,-31,62,-62,93,-93, 124, -124, 155, -155, 186, -186]
#  [0, 31, -31, 62, -62]
step = 3  # degrees per frame

frames_deg = []
pause_frames = int(0.1 * 30) # 0.1 second pause at 30 FPS  # added line

for start, end in zip(object_spin_angle_deg[:-1], object_spin_angle_deg[1:]):
    if start < end:
        segment = list(range(start, end, step))
    else:
        segment = list(range(start, end, -step))

    frames_deg.extend(segment)
    #Add the pausese in the frame cycle at the end of each segment   added
    for _ in range(pause_frames):   # added line
        frames_deg.append(end)      # added line

# include last value
# add the last  value  added line
frames_deg.append(object_spin_angle_deg[-1])
# frames_deg

In [ ]:
# for start, end in zip(object_spin_angle_deg[:-1], object_spin_angle_deg[1:]):
#   print(start, end)

In [ ]:
SVG_NS = "http://www.w3.org/2000/svg"

INPUT_SVG = "scene.svg"
FRAMES_DIR = "frames"
VIDEO_NAME = "video_v6.mp4"

FPS = 30
TOTAL_FRAMES = 600

object_spin_angle = [0,31,-31,62,-62,93,-93, 124, -124, 155, -155, 186, -186]
object_spin_angle_deg = [0,31,-31,62,-62,93,-93,124,-124,155,-155,186,-186]

# object_spin_angle = [0,31,-31,62,-62]
# object_spin_angle_deg = [0,31,-31,62,-62]

object_spin_angle_rad = [angle * math.pi / 180 for angle in object_spin_angle_deg]


def update_object_transform(tree, final_x, final_y, scale_factor, object_spin_angle,
                            bell_pivot_local_x, bell_pivot_local_y):
    root = tree.getroot()

    # The order of transforms matters. SVG applies right-to-left.
    # We want to:
    # 1. Translate the object so its local pivot is at (0,0) -> translate(-bell_pivot_local_x, -bell_pivot_local_y)
    # 2. Rotate around (0,0) -> rotate(object_spin_angle)
    # 3. Scale around (0,0) -> scale(scale_factor)
    # 4. Translate the scaled and rotated object so its pivot (which was at (0,0))
    #    is now at (final_x, final_y) in the scene. -> translate(final_x, final_y)

    # Combining these right-to-left in the transform string:
    transform_string = (
        f"translate({final_x} {final_y}) "
        f"scale({scale_factor}) "
        f"rotate({object_spin_angle}) "
        f"translate({-bell_pivot_local_x} {-bell_pivot_local_y})"
    )

    for el in root.iter(f"{{{SVG_NS}}}g"):
        if el.get("id") == "movable_object":
            el.set("transform", transform_string)
    return tree

def generate_frames():
    os.makedirs(FRAMES_DIR, exist_ok=True)

    # Orbit parameters: The point on the background around which the object's pivot will orbit
    orbit_center_x = 745 #737
    orbit_center_y = 800 #825  #850 #708
    orbit_radius = 0


    for i, orbit_angle in enumerate(frames_deg):
        final_x = orbit_center_x + orbit_radius
        final_y = orbit_center_y + orbit_radius

        # Object's own rotation and scaling
        scale_factor = 0.7 #2.3
        # object_spin_angle = orbit_angle * 3 # Rotate 3 degrees per frame around its local pivot
        orbit_angle

        tree = ET.parse(INPUT_SVG)
        tree = update_object_transform(
            tree,
            final_x,
            final_y,
            scale_factor,
            orbit_angle,
            bell_pivot_local_x, # Global variable calculated from Bell.svg viewBox
            bell_pivot_local_y  # Global variable calculated from Bell.svg viewBox
        )

        svg_path = os.path.join(FRAMES_DIR, f"frame_{i:04}.svg")
        png_path = os.path.join(FRAMES_DIR, f"frame_{i:04}.png")

        tree.write(svg_path, encoding="utf-8", xml_declaration=True)

        # Convert SVG → PNG
        cairosvg.svg2png(
        url=svg_path,
        write_to=png_path,
        background_color="white"
        )

def create_video():
    images = sorted(glob.glob(f"{FRAMES_DIR}/frame_*.png"))

    if not images:
        print("No frames found to create video.")
        return

    frame = cv2.imread(images[0])
    height, width, _ = frame.shape

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_v6 = cv2.VideoWriter(VIDEO_NAME, fourcc, FPS, (width, height))

    for img_path in images:
        frame = cv2.imread(img_path)
        video_v6.write(frame)

    video_v6.release()


if __name__ == "__main__":
    generate_frames()
    create_video()
    print("Video created:", VIDEO_NAME)


Video created: video_v6.mp4
